# Condition Monitoring: Full Diagnosis Pipeline

This notebook demonstrates the integrated diagnosis pipeline that combines:
- FFT spectral analysis
- Power Spectral Density (Welch)
- STFT time-frequency analysis
- Bearing fault detection
- ISO 10816 severity assessment
- Anomaly detection (OneClassSVM)

In [ ]:
import numpy as np
from pathlib import Path
import sys, os

project_root = Path('.').resolve().parent
sys.path.insert(0, str(project_root / 'src'))
os.chdir(project_root)

## Step 1: Create Test Signals (Healthy + Faulty)

In [ ]:
fs = 10000
duration = 2.0
t = np.arange(0, duration, 1/fs)
rpm = 1800
f_shaft = rpm / 60.0

# Healthy signal: shaft + noise
healthy = (
    0.3 * np.sin(2 * np.pi * f_shaft * t)
    + 0.1 * np.sin(2 * np.pi * 2 * f_shaft * t)
    + 0.15 * np.random.randn(len(t))
)

# Faulty signal: shaft + bearing fault + noise
bpfo = 120.0
faulty = (
    0.3 * np.sin(2 * np.pi * f_shaft * t)
    + 0.5 * np.sin(2 * np.pi * bpfo * t)
    + 0.25 * np.sin(2 * np.pi * 2 * bpfo * t)
    + 0.1 * np.sin(2 * np.pi * 3 * bpfo * t)
    + 0.2 * np.random.randn(len(t))
)

print(f'Created healthy signal: RMS = {np.sqrt(np.mean(healthy**2)):.4f}')
print(f'Created faulty signal:  RMS = {np.sqrt(np.mean(faulty**2)):.4f}')

## Step 2: Run Full Diagnosis Pipeline

In [ ]:
from predictive_maintenance_mcp.decision_support import diagnose_vibration

# Diagnose the healthy signal
healthy_diag = diagnose_vibration(
    signal=healthy, fs=fs, rpm=rpm,
    machine_class='II', signal_unit='g'
)

print('=== HEALTHY SIGNAL DIAGNOSIS ===')
print(f'Overall: {healthy_diag.get("overall_condition", "N/A")}')
print(f'Confidence: {healthy_diag.get("confidence", "N/A")}')
print(f'ISO Zone: {healthy_diag.get("iso_severity", {}).get("zone", "N/A")}')
if healthy_diag.get('recommendations'):
    print('Recommendations:')
    for r in healthy_diag['recommendations'][:3]:
        print(f'  - {r}')

In [ ]:
# Diagnose the faulty signal
faulty_diag = diagnose_vibration(
    signal=faulty, fs=fs, rpm=rpm,
    bearing_id='6205',
    machine_class='II', signal_unit='g'
)

print('=== FAULTY SIGNAL DIAGNOSIS ===')
print(f'Overall: {faulty_diag.get("overall_condition", "N/A")}')
print(f'Confidence: {faulty_diag.get("confidence", "N/A")}')
print(f'ISO Zone: {faulty_diag.get("iso_severity", {}).get("zone", "N/A")}')

# Bearing faults found
bearing_results = faulty_diag.get('bearing_faults', [])
if bearing_results:
    print(f'\nBearing faults detected ({len(bearing_results)}):')
    for fault in bearing_results:
        if fault.get('detected'):
            print(f'  {fault["fault_type"]}: confidence={fault.get("confidence", "?")}')

if faulty_diag.get('recommendations'):
    print('\nRecommendations:')
    for r in faulty_diag['recommendations'][:5]:
        print(f'  - {r}')

## Step 3: Compare Spectral Signatures

In [ ]:
from predictive_maintenance_mcp.signal_processing import compute_psd

healthy_psd = compute_psd(healthy, fs, nperseg=1024, num_peaks=5)
faulty_psd = compute_psd(faulty, fs, nperseg=1024, num_peaks=5)

print('Healthy PSD peaks:')
for p in healthy_psd['top_peaks']:
    print(f'  {p["frequency_hz"]:8.2f} Hz  ({p["magnitude_db"]:+.1f} dB)')

print(f'\nFaulty PSD peaks:')
for p in faulty_psd['top_peaks']:
    print(f'  {p["frequency_hz"]:8.2f} Hz  ({p["magnitude_db"]:+.1f} dB)')

print(f'\nPower ratio (faulty/healthy): {faulty_psd["total_power"] / healthy_psd["total_power"]:.2f}x')

## Step 4: ISO 10816 Comparison

In [ ]:
from predictive_maintenance_mcp.diagnostics import assess_vibration_severity

h_sev = assess_vibration_severity(healthy, fs, machine_class='II', signal_unit='g')
f_sev = assess_vibration_severity(faulty, fs, machine_class='II', signal_unit='g')

print(f'{"Signal":>10s} | {"Zone":>6s} | {"Severity":>15s} | {"Vel RMS (mm/s)":>14s}')
print('-' * 55)
print(f'{"Healthy":>10s} | {h_sev["zone"]:>6s} | {h_sev["severity"]:>15s} | {h_sev["velocity_rms_mm_s"]:>14.2f}')
print(f'{"Faulty":>10s} | {f_sev["zone"]:>6s} | {f_sev["severity"]:>15s} | {f_sev["velocity_rms_mm_s"]:>14.2f}')

## Summary

This notebook demonstrated the complete condition monitoring workflow:

1. **Signal comparison**: Healthy vs faulty signals side-by-side
2. **Integrated diagnosis**: `diagnose_vibration()` combines FFT + PSD + STFT + bearing + ISO
3. **Spectral signatures**: PSD reveals fault-related frequency components
4. **ISO severity**: Quantitative assessment against international standards

### Architecture

All processing follows the ISO 13374 6-block architecture:
- **Block 1** (`signal_acquisition`): Load and cache signals
- **Block 2** (`signal_processing`): Spectral analysis
- **Blocks 3-4** (`diagnostics`): Fault detection + severity
- **Block 6** (`decision_support`): Integrated diagnosis